<a href="https://colab.research.google.com/github/brandywii/dsa405-project/blob/main/DSA405_002_FA26_P2_bdwilki2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
###############################################################################################################################################################
# Author: Brandy Wilkinson
#         DSA 405
#         Fall 2026
#
# Description:
#         This script extracts and processes data from eCFR (data source 1)
#
# Question: Which primary raw fast-food ingredients contain the highest frequencies and concentrations of glyphosate,
#         and do those levels compare to the EPA Maximum Residue Limits (MRLs).
#         Health conscious consumers currently cannot easily access or compare pesticide surveillance data against federal regulatory tolerances.
#         Answering this question reveals whether the common fast food agricultural inputs stay within safe regulatory limits, providing consumers with transparent, official data-backed insights into thier true glyphosate exposure everytime they dine out.
#         [Note] Legal limit of a pesticide left on food is called a tolerance and these tolerances are established by the U.S. EPA (published at eCFR).
#
# Issues:
#        In-progress, Need to complete script for USDA Pesticide Data Program (PDP). Data is filterable and downloab=dable via csv file
# ###############################################################################################################################################################


# Load relevant packages
import io
import pandas as pd
import requests

URL = "https://www.ecfr.gov/api/renderer/v1/content/enhanced/2026-09-01/title-40?chapter=I&subchapter=E&part=180&subpart=C&section=180.364"

UA = {"User-Agent": "DSA405-student P1 (NC State student)"}

# Request data
html_data = requests.get(URL, headers=UA, timeout=30).text

# Put data into a table
data_tables = pd.read_html(io.StringIO(html_data))

print(f"Num of tables found: {len(data_tables)}")

# Tag the source tables before combining to flag GMO vs Conventional accurately
# Table 0 represents standard/conventional tolerances
df_conv = data_tables[0].copy()
df_conv["Is_GMO"] = False

# Table 1 represents specialized/GMO-associated traits
df_gmo = data_tables[1].copy()
df_gmo["Is_GMO"] = True

# Combine standard/conventional and GMO tables
df_ecfr = pd.concat([df_conv, df_gmo], ignore_index=True)

# Coalesce the two separate tolerance columns into a single unified one called `Tolerance_ppm`
df_ecfr["Tolerance_ppm"] = df_ecfr["Parts per million"].combine_first(
    df_ecfr["Parts per Million"]
)

# Drop the old split columns
df_ecfr = df_ecfr.drop(columns=["Parts per million", "Parts per Million"])

# Reorder columns so 'Is_GMO' is the third column (index 2)
df_ecfr = df_ecfr[["Commodity", "Tolerance_ppm", "Is_GMO"]]

# Strip any whitespace or hidden formatting from commodity text
df_ecfr["Commodity"] = df_ecfr["Commodity"].str.strip()

# Display eCFR table shape
print(f"eCFR table rows and columns: {df_ecfr.shape}")

# Display all the rows of the dataframe called 'df_ecfr'
pd.set_option("display.max_rows", None)

# Display as HTML grid, use variable over print()
df_ecfr


Num of tables found: 2
eCFR table rows and columns: (158, 3)


,Commodity,Tolerance_ppm,Is_GMO
0,Acerola,0.20,False
1,"Alfalfa, seed",0.50,False
2,"Almond, hulls",25.00,False
3,Aloe vera,0.50,False
4,Ambarella,0.20,False
5,"Animal feed, nongrass, group 18",400.00,False
6,"Artichoke, globe",0.20,False
7,Asparagus,0.50,False
8,Atemoya,0.20,False
9,Avocado,0.20,False


#1. My Systematic Audit

**Data types: dtype**
`Commodity` column data was loaded as an object which is correct for strings.
`Tolerance_ppm` column loaded then converted to a float which is correct for residue limits.
`Is_GMO` column was loaded as boolean which is correct for binary indicator flags.

**Missing Values(dropna=False) retains all Na's**
`Commodity` column contains 0 missing values.
`Tolerance_ppm` column contains 0 missing values after two tables were combined into one. One table was for conventionally grown commodities while the second table contained tolerance levels for Genetically Modified Organism (GMO) grown commodities.
`Is_GMO` column contains 0 missing values. This column was not part of the original dataset; I included it to clearly distinguish between the two tables since one was conventional and one GMO.

**Number of distinct values**
`Commodity` contains 158 distinct commodities which aligns with total row count.
`Tolerance_ppm` contains 42 distinct numerical thresholds (aka minimum residue limits)
`Is_GMO` contains 2 distinct values (true or false)

**Numerical Ranges derived from `Tolerance_ppm` column****
Minimum value: 0.10
Maximum value: 400.00
Range: 399.90 (max - min)

Categorical values derived from `Is_GMO`**
True for GMO, False for Conventionally grown

**Defective data check**
Sentinel values: Data was meticulously checked for numerical anomolies such as 999 or "N/A" and none were found.

Subtotal/Total rows mixed with observations: None found

Leading zeros in identifiers: Data checked for padding/leading zeros and none were detected.

Character encoding errors: Data checked for garbled text as a result of faulty file encodings from when html data was extracted and none were detected.

Duplicated keys: Data checked for duplicated commodity key names and all 158 commodities are unique and map to one row. No duplicates detected and all 158 are unique/distinct.

Inconsistent date formats: This dataset does not contain any dates so none were detected.

Numerical values as text: all numerical data was successfully parsed as float datatypes and no strings were detected in the only numerical column, `Tolerance_ppm`

Final audit findings: eCFR regulatory data on US EPA Minimum Residue Limits (MRLs) is genuinely clean and should earn the top score for this criterion.

#2. My Data Dictionary

| **Field** | **Commodity** | **Tolerance_ppm** | **Is_GMO**
|---|---|---|---|
| **Variable name** | Commodity | Tolerance_ppm | Is_GMO |
| **Type** | string | float | boolean |
| **Units** | N/A | parts per million (ppm) | N/A |
| **Factor levels** | 158 distinct levels | continuous numeric | True or False |
| **Valid range** | All agricultural commodity | 0.00 to infinity ppm. (0.10 to 400.00 in this dataset) | True or False |
| **Missingness** | 0 missing values. Missing means unrecorded commodity entry | 0 missing values. Missing means No Tolerance limit established | 0 missing values. Missing values are impossible since I added this column and the two tables are either conventional or GMO |
| **Notes** | Whitespace stripped from strings. Commodities are either conventional or GMO | Legal tolerance limits established by US EPA under eCFR Title 40 | Custom flag added to distinguish std conventional (false) from GMO commodities (true) |


#3. My cleaning log

| # | Column | Change made | Rows/cells affected | Why | What is lost |
|---|---|---|---|---|---|
| 1 | `Commodity` | Stripped whitespace | 158 cells | Leading & trailing spaces introduced formatting issues in merged table rows| Nothing |
| 2 | `Parts per million` & `Parts per Million`| Combined two tolerance columns into one called `Tolerance_ppm` | 158 rows | API render cut the table two seperate case-sensitive columns which prevented analysis | Nothing |
| 3 | `Tolerance_ppm` | Redundant split columns were dropped (Parts per million & Parts per Million) | 316 cells (2 columns x 158 rows) | Removing duplicate column streamlines df for the merge | Nothing; underlying numeric vals are preserved in `Tolerance_ppm` |
| 4 | `Commodity`| Columns wewre reordered; placed `Tolerance_ppm` 2nd and created `Is_GMO` as third column | 158 rows | Column order helps with readability and project structure| Nothing; supports structural presentation/order |
| 5 | `Is_GMO` | included binary indicator flag (True = GMO) (False = conventional) | 158 rows | Used to distinguish between two source tables since one represents standard crops while other represents GMO traits | Nothing; this is a custom column I added to help the user |
| 6 | `Is_GMO` | missing rows filled with a boolean value | 17 cells = True (GMO) while 141 cells = False (Standard)| Tagging rows before table concatenation will result in correct classification without any weird gaps; improves readability | Nothing |
| 7 | `Commodity` | Searched for missing values across the primary key names | 0 rows affected | Validated that no commodites were lost or corrupted during string processing | Nothing |
| 8 | `Tolerance_ppm`| Searched for missing numerical values in `Tolerance_ppm` | 0 rows affected | Verify that coalescing the split columns didnt leave any 0 null gaps| Nothing |
| 9 | `Is_GMO` | Searched for missing values across binary flags | 0 rows affected | Confirms that all records are tagged as standard or GMO | Nothing |
| 10 | `Is_GMO` | Reordered column position to index 2/ 3rd column | 158 rows | Ensure schema is aligned with my desired layout | Nothing |


#4. My row and column accounting

```
Rows in raw file                      _158____
Rows removed as exact duplicates      __0___
Rows removed as near-duplicates       __0__
Rows removed for other reasons        __0___   (each one logged above)
Rows in cleaned file                  _158___

Columns in raw file                   __3___
Columns dropped                       __2__see [note 1]    (and why)
Columns added                         __2__
Columns in cleaned file               __3___

[note 1]  Parts per million and Parts per Million columns were dropped because they were unified into a single column called `Tolerance_ppm`. One was for standard commodites and the other was for GMO. I combined them into one column and added the `Is_GMO` column to help the untrained user distinguish between the two; this is a must because you have to read the fine print to understand the differences and its not really obvious looking at the orignal dataset. My intent was to make it as easy as possible for any untrained user to see and understand.

**Reconciliation Update**
Rows: 158 rows - 0 - 0 - 0 = 158 rows
Columns: 3 initial - 2 dropped + 2 added = 3 final columns
All rows, columns, and changes are logged and accounted for with no mismatches detected.
```

#5. The Provenance Brief by Brandy Wilkinson

The dataset contains the official U.S. Environmental Protection Agency (EPA) minimum residue limits (MRLs). MRLs are established tolerances for various pesticide chemicals on agricultural commodities consumed in the U.S. The dataset was compiled from the Electronic Code of Federal Regulations (eCFR) which is used to legally establish maximum residue limits—measured in parts per million—to protect human and environmental health.

The dataset is comprised of 158 total raw agricultural commodities and established legal limits of chemical residue. There are two tables provided: one for standard conventional commodities and a second for genetically modified organism (GMO) commodities.

The values establish legal federal safety limits, allow for comparison of regulatory thresholds between standard and GMO agricultural commodities, and serve as a baseline benchmark for safety compliance.

The values do not reflect real-world pesticide residue limits for products on store shelves, they do not track consumer health outcomes after product consumption, and they do not consider pesticide application practices. Since these limits represent regulatory ceilings rather than actual measurements, they are unable to evaluate consumption exposure independently.

# P2- Audit, Cleaning Log & Provenance Brief Rubric

| # | Criterion | Wt | Self Score |  
|---|---|---|---|
| 1 | **Diagnosis & data dictionary** | ×2 | 4- Excellent |
| 2 | **Cleaning execution** | ×2 | 4- Excellent |
| 3 | **Cleaning log** | ×3 | 4- Excellent |
| 4 | **Provenance Brief** | ×1 | 4- Excellent |
| 5 | **Tidy structure & reproducibility** | x2 | 4- Excellent |

# AI Use Policy

No AI tools were used to complete this assignment